# Learning Rate Schedules

## Learning Objectives
1. Understand why fixed learning rates fail and how schedules improve convergence.
2. Implement warmup + cosine annealing from scratch in NumPy and visualise the LR curve.
3. Compare StepLR, CosineAnnealingLR, OneCycleLR, and ReduceLROnPlateau in PyTorch.
4. Apply warmup+cosine for transformer fine-tuning, cyclical LR, and the LR range finder.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: Warmup + Cosine Annealing (NumPy Visualisation)

In [ ]:
# Warmup: ramp LR from 0 to peak over warmup_steps to stabilise early training.
# Cosine annealing: smoothly decay LR following a cosine curve.
# Combined: widely used for transformer fine-tuning (BERT, GPT, T5).

def warmup_cosine_schedule(step, warmup_steps, total_steps, lr_max, lr_min=0.0):
    '''Compute LR at a given step with linear warmup + cosine decay.
    Args:
        step:         current training step (0-indexed)
        warmup_steps: steps to ramp from 0 to lr_max
        total_steps:  total training steps
        lr_max:       peak learning rate
        lr_min:       minimum LR at end of cosine decay
    Returns:
        learning rate for this step
    '''
    if step < warmup_steps:
        # Linear warmup
        return lr_max * step / max(1, warmup_steps)
    # Cosine decay after warmup
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine_factor = 0.5 * (1 + np.cos(np.pi * progress))
    return lr_min + (lr_max - lr_min) * cosine_factor


def step_schedule(step, lr_init, step_size, gamma):
    '''StepLR: multiply by gamma every step_size steps.'''
    return lr_init * (gamma ** (step // step_size))


TOTAL_STEPS  = 1000
WARMUP_STEPS = 100
LR_MAX       = 3e-4

steps    = np.arange(TOTAL_STEPS)
lr_wcos  = [warmup_cosine_schedule(s, WARMUP_STEPS, TOTAL_STEPS, LR_MAX)
            for s in steps]
lr_step  = [step_schedule(s, LR_MAX, 250, 0.5) for s in steps]
lr_const = [LR_MAX] * TOTAL_STEPS

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, lr_wcos,  label='Warmup+Cosine (transformer standard)', lw=2)
ax.plot(steps, lr_step,  label='StepLR (gamma=0.5, step=250)',         lw=2, linestyle='--')
ax.plot(steps, lr_const, label='Constant LR',                          lw=1, linestyle=':')
ax.axvline(WARMUP_STEPS, color='gray', linestyle=':', alpha=0.6, label='End of warmup')
ax.set_xlabel('Training Step'); ax.set_ylabel('Learning Rate')
ax.set_title('Learning Rate Schedule Comparison')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/lr_schedules.png', dpi=72)
print('Schedule plot saved to /tmp/lr_schedules.png')
print(f'Warmup+Cosine peak at step {WARMUP_STEPS}: LR={lr_wcos[WARMUP_STEPS]:.2e}')
print(f'Warmup+Cosine final LR: {lr_wcos[-1]:.2e}')


## Level 2: StepLR / CosineAnnealingLR / OneCycleLR / ReduceLROnPlateau in PyTorch

In [ ]:
# Compare four PyTorch schedulers on the same training task.
# Each scheduler is run for the same number of epochs for fair comparison.

import time

torch.manual_seed(0)
X_sched = torch.randn(1000, 32, device=device)
y_sched = torch.randint(0, 6, (1000,), device=device)
sched_ds  = TensorDataset(X_sched[:800], y_sched[:800])
sched_val = TensorDataset(X_sched[800:], y_sched[800:])
sched_trl = DataLoader(sched_ds,  batch_size=32, shuffle=True, drop_last=True)
sched_vll = DataLoader(sched_val, batch_size=128)


def make_model():
    return nn.Sequential(
        nn.Linear(32, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 6)
    ).to(device)


N_EPOCHS = 40
LR_INIT  = 1e-2

scheduler_configs = {
    'StepLR':              lambda opt: torch.optim.lr_scheduler.StepLR(
                                        opt, step_size=10, gamma=0.5),
    'CosineAnnealingLR':   lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(
                                        opt, T_max=N_EPOCHS, eta_min=1e-5),
    'OneCycleLR':          lambda opt: torch.optim.lr_scheduler.OneCycleLR(
                                        opt, max_lr=LR_INIT,
                                        steps_per_epoch=len(sched_trl),
                                        epochs=N_EPOCHS, pct_start=0.3),
    'ReduceLROnPlateau':   lambda opt: torch.optim.lr_scheduler.ReduceLROnPlateau(
                                        opt, mode='min', patience=3, factor=0.5),
}

sched_results = {}
for name, sched_fn in scheduler_configs.items():
    model_s = make_model()
    opt_s   = torch.optim.SGD(model_s.parameters(), lr=LR_INIT, momentum=0.9)
    sched   = sched_fn(opt_s)
    is_onecycle  = (name == 'OneCycleLR')
    is_plateau   = (name == 'ReduceLROnPlateau')
    val_losses = []
    for epoch in range(N_EPOCHS):
        model_s.train()
        for xb, yb in sched_trl:
            try:
                opt_s.zero_grad()
                nn.CrossEntropyLoss()(model_s(xb), yb).backward()
                opt_s.step()
                if is_onecycle: sched.step()   # OneCycleLR steps per batch
            except RuntimeError as exc:
                if 'out of memory' in str(exc).lower():
                    torch.cuda.empty_cache(); continue
                raise
        model_s.eval()
        with torch.no_grad():
            vl = sum(
                nn.CrossEntropyLoss()(model_s(xb), yb).item() * len(xb)
                for xb, yb in sched_vll
            ) / len(sched_val)
        val_losses.append(vl)
        if not is_onecycle:
            if is_plateau: sched.step(vl)
            else:          sched.step()
    sched_results[name] = {'final_val_loss': val_losses[-1],
                           'best_val_loss':  min(val_losses)}

print(f'{'Scheduler':<25} {'Final Loss':>12} {'Best Loss':>12}')
print('-' * 52)
for n, r in sched_results.items():
    print(f'{n:<25} {r["final_val_loss"]:>12.4f} {r["best_val_loss"]:>12.4f}')


## Real-World Example 1: Warmup + Cosine Schedule for Transformer Fine-Tuning

In [ ]:
# Implement the Hugging Face-style warmup+cosine schedule as a PyTorch LambdaLR.
# This is the standard schedule used when fine-tuning BERT/GPT/T5.

torch.manual_seed(1)


class SmallTransformer(nn.Module):
    '''Tiny encoder-only transformer for sequence classification.'''
    def __init__(self, vocab_size=100, d_model=64, n_heads=4, n_layers=2,
                 max_seq=16, n_classes=4):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model)
        self.pos_enc  = nn.Embedding(max_seq, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True
        )
        self.encoder  = nn.TransformerEncoder(enc_layer, n_layers)
        self.head     = nn.Linear(d_model, n_classes)

    def forward(self, x):
        pos  = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        emb  = self.embed(x) + self.pos_enc(pos)
        enc  = self.encoder(emb)
        return self.head(enc[:, 0, :])   # CLS-token classification


transformer_model = SmallTransformer().to(device)

# Synthetic token sequence dataset
X_tok = torch.randint(0, 100, (800, 16), device=device)
y_tok = torch.randint(0, 4,   (800,),    device=device)
tok_ds  = TensorDataset(X_tok[:640], y_tok[:640])
tok_val = TensorDataset(X_tok[640:], y_tok[640:])
tok_trl = DataLoader(tok_ds,  batch_size=32, shuffle=True, drop_last=True)
tok_vll = DataLoader(tok_val, batch_size=64)

N_EPOCHS_TF  = 30
WARMUP_RATIO = 0.1   # 10% of total steps for warmup
STEPS_TOTAL  = N_EPOCHS_TF * len(tok_trl)
WARMUP_STEPS_TF = int(WARMUP_RATIO * STEPS_TOTAL)

opt_tf = torch.optim.AdamW(transformer_model.parameters(), lr=3e-4, weight_decay=0.01)


def get_lr_lambda(current_step):
    '''LambdaLR function: returns LR multiplier for current_step.'''
    if current_step < WARMUP_STEPS_TF:
        return float(current_step) / float(max(1, WARMUP_STEPS_TF))
    progress = float(current_step - WARMUP_STEPS_TF) / float(
        max(1, STEPS_TOTAL - WARMUP_STEPS_TF))
    return max(0.0, 0.5 * (1.0 + np.cos(np.pi * progress)))


scheduler_tf = torch.optim.lr_scheduler.LambdaLR(opt_tf, get_lr_lambda)
val_accs_tf  = []
global_step  = 0

for epoch in range(N_EPOCHS_TF):
    transformer_model.train()
    for xb, yb in tok_trl:
        opt_tf.zero_grad()
        nn.CrossEntropyLoss()(transformer_model(xb), yb).backward()
        torch.nn.utils.clip_grad_norm_(transformer_model.parameters(), 1.0)
        opt_tf.step()
        scheduler_tf.step()
        global_step += 1
    transformer_model.eval()
    with torch.no_grad():
        correct = sum(
            (transformer_model(xb).argmax(1)==yb).sum().item()
            for xb, yb in tok_vll
        )
    val_accs_tf.append(correct / len(tok_val))

print(f'Warmup+Cosine transformer fine-tuning val acc: {val_accs_tf[-1]:.4f}')
print(f'Warmup steps: {WARMUP_STEPS_TF} / {STEPS_TOTAL} total ({WARMUP_RATIO*100:.0f}%)')
print('Standard recipe: AdamW + warmup+cosine + gradient clipping (norm=1.0)')


## Real-World Example 2: Cyclical Learning Rates (CLR)

In [ ]:
# Cyclical LR (Smith 2015): cycle LR between base_lr and max_lr.
# Avoids saddle points: occasional LR spikes escape local minima.
# PyTorch CyclicLR supports 'triangular', 'triangular2', and 'exp_range' modes.

torch.manual_seed(2)

model_clr  = make_model()
opt_clr    = torch.optim.SGD(model_clr.parameters(), lr=1e-4, momentum=0.9)
clr_sched  = torch.optim.lr_scheduler.CyclicLR(
    opt_clr,
    base_lr=1e-4,
    max_lr=1e-2,
    step_size_up=len(sched_trl) * 4,   # 4 epochs to go from base to max
    mode='triangular2',                # halve max_lr each cycle
    cycle_momentum=True,               # cycle momentum opposite to LR
    base_momentum=0.85,
    max_momentum=0.95,
)

clr_lrs   = []
clr_losses = []

for epoch in range(40):
    model_clr.train()
    epoch_loss = 0.0
    n_batches  = 0
    for xb, yb in sched_trl:
        opt_clr.zero_grad()
        loss = nn.CrossEntropyLoss()(model_clr(xb), yb)
        loss.backward()
        opt_clr.step()
        clr_sched.step()                     # CLR steps per batch
        clr_lrs.append(opt_clr.param_groups[0]['lr'])
        epoch_loss += loss.item(); n_batches += 1
    clr_losses.append(epoch_loss / n_batches)

model_clr.eval()
with torch.no_grad():
    correct = sum(
        (model_clr(xb).argmax(1)==yb).sum().item()
        for xb, yb in sched_vll
    )
print(f'CLR (triangular2) val accuracy: {correct/len(sched_val):.4f}')
print(f'LR range observed: [{min(clr_lrs):.2e}, {max(clr_lrs):.2e}]')
print(f'Final training loss: {clr_losses[-1]:.4f}')
print('triangular2 halves the amplitude each cycle — progressively finer search.')


## Real-World Example 3: LR Range Finder (Smith's LR Test)

In [ ]:
# LR Range Finder: sweep LR exponentially from very small to very large.
# Plot training loss vs LR — choose LR where loss decreases fastest
# (the 'elbow' in the curve), ~10x below the divergence point.

torch.manual_seed(3)

model_lrf = make_model()
opt_lrf   = torch.optim.SGD(model_lrf.parameters(), lr=1e-7, momentum=0.9)

LR_MIN   = 1e-7
LR_MAX_F = 1.0
N_STEPS  = len(sched_trl) * 5   # 5 epochs of data
LR_MULT  = (LR_MAX_F / LR_MIN) ** (1.0 / N_STEPS)   # exponential step

lrf_lrs    = []
lrf_losses = []
smoothed_loss = None
BETA = 0.98   # smoothing factor

loader_cycle = iter(sched_trl)
best_loss = float('inf')

for step in range(N_STEPS):
    try:
        xb, yb = next(loader_cycle)
    except StopIteration:
        loader_cycle = iter(sched_trl)
        xb, yb = next(loader_cycle)

    opt_lrf.zero_grad()
    loss = nn.CrossEntropyLoss()(model_lrf(xb), yb)
    loss.backward()
    opt_lrf.step()

    # Exponentially increase LR
    current_lr = LR_MIN * (LR_MULT ** step)
    for g in opt_lrf.param_groups: g['lr'] = current_lr

    # Smooth the loss to reduce noise
    if smoothed_loss is None:
        smoothed_loss = loss.item()
    else:
        smoothed_loss = BETA * smoothed_loss + (1 - BETA) * loss.item()

    lrf_lrs.append(current_lr)
    lrf_losses.append(smoothed_loss)

    # Stop if loss explodes (>4x best seen so far)
    if smoothed_loss < best_loss: best_loss = smoothed_loss
    if smoothed_loss > 4 * best_loss: break

# Find optimal LR: steepest descent
lrf_arr = np.array(lrf_losses)
lr_arr  = np.array(lrf_lrs)
if len(lrf_arr) > 5:
    gradients    = np.gradient(lrf_arr)
    optimal_idx  = gradients.argmin()
    optimal_lr   = lr_arr[optimal_idx]
else:
    optimal_lr   = LR_MIN * 10

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lr_arr[:len(lrf_losses)], lrf_losses)
ax.axvline(optimal_lr, color='red', linestyle='--',
           label=f'Suggested LR: {optimal_lr:.2e}')
ax.set_xscale('log'); ax.set_xlabel('Learning Rate (log scale)')
ax.set_ylabel('Smoothed Loss'); ax.set_title('LR Range Finder')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/lr_finder.png', dpi=72)
print(f'Suggested learning rate: {optimal_lr:.2e}')
print('Rule of thumb: use LR at the steepest loss descent, ~10x before divergence.')
print('LR finder plot saved to /tmp/lr_finder.png')
